# Auditoria das populações interpoladas

In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

PASTA_ATUAL = Path.cwd()
PASTA_POP = PASTA_ATUAL / "População"

OUTFILE = PASTA_POP / "auditoria_populacao_popnova.xlsx"

ANOS_POPULACAO = list(range(2000, 2025))
ANOS_POPNOVA = list(range(2001, 2025))
ANOS_CASOS = list(range(2001, 2025))

LIMIAR_VARIACAO_ANUAL_PCT = 20
LIMIAR_CAGR_PCT = 12
LIMIAR_POP_PEQUENA = 10_000
LIMIAR_INCIDENCIA_EXTREMA = 1_000
MIN_CASOS_INCIDENCIA_EXTREMA = 20
MIN_POP_INCIDENCIA_EXTREMA = 10_000

USAR_CAGR_POPULACAO_XLSX = True
USAR_CAGR_POPNOVA = False
USAR_ALERTA_INCIDENCIA = True


def sem_acento(txt):
    if pd.isna(txt):
        return ""
    txt = str(txt).strip()
    txt = unicodedata.normalize("NFKD", txt)
    return "".join(ch for ch in txt if not unicodedata.combining(ch))


def norm_text(txt):
    txt = sem_acento(txt).upper()
    txt = re.sub(r"[^A-Z0-9 ]+", " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt


def normalizar_nome_arquivo(nome):
    nome = sem_acento(nome).lower()
    nome = re.sub(r"[^a-z0-9]+", "", nome)
    return nome


def procurar_arquivo(possiveis_nomes, pastas):
    if isinstance(possiveis_nomes, str):
        possiveis_nomes = [possiveis_nomes]

    nomes_norm = [normalizar_nome_arquivo(n) for n in possiveis_nomes]

    for pasta in pastas:
        for nome in possiveis_nomes:
            caminho = pasta / nome
            if caminho.exists():
                return caminho

    for pasta in pastas:
        if pasta.exists():
            for arq in pasta.rglob("*"):
                if arq.is_file():
                    if normalizar_nome_arquivo(arq.name) in nomes_norm:
                        return arq

    raise FileNotFoundError(
        "Não encontrei nenhum destes arquivos: "
        + ", ".join(possiveis_nomes)
        + "\nPastas procuradas: "
        + ", ".join(str(p) for p in pastas)
    )


ARQ_POPULACAO = procurar_arquivo(
    ["População.xlsx", "Populacao.xlsx"],
    [PASTA_POP, PASTA_ATUAL]
)

ARQ_POPNOVA = procurar_arquivo(
    ["POPNOVA_fim.xlsx", "POPNOVA.xlsx", "popnova_fim.xlsx"],
    [PASTA_POP, PASTA_ATUAL]
)

ARQ_CASOS_SP = procurar_arquivo(
    ["Casos-TB-SP-NOVO.csv"],
    [PASTA_ATUAL, PASTA_POP]
)

ARQ_CASOS_RJ = procurar_arquivo(
    ["Casos-TB-RJ-NOVO.csv"],
    [PASTA_ATUAL, PASTA_POP]
)

print("Arquivos encontrados:")
print("População:", ARQ_POPULACAO)
print("POPNOVA:", ARQ_POPNOVA)
print("Casos SP:", ARQ_CASOS_SP)
print("Casos RJ:", ARQ_CASOS_RJ)
print("Saída:", OUTFILE)


def cod6_from_any(x):
    if pd.isna(x):
        return np.nan
    s = re.sub(r"\D", "", str(x))
    if len(s) >= 7:
        return s[:6]
    if len(s) == 6:
        return s
    return s.zfill(6) if s else np.nan


def clean_number(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return x
    s = str(x).strip()
    if s in ["", "-", "–", "—", "...", "nan", "NaN"]:
        return 0
    s = s.replace(".", "").replace(",", ".")
    return pd.to_numeric(s, errors="coerce")


def achar_aba(excel_path, nomes_possiveis):
    xls = pd.ExcelFile(excel_path)
    abas = xls.sheet_names

    abas_norm = {norm_text(a): a for a in abas}

    for nome in nomes_possiveis:
        nome_norm = norm_text(nome)
        if nome_norm in abas_norm:
            return abas_norm[nome_norm]

    for aba in abas:
        aba_norm = norm_text(aba)
        for nome in nomes_possiveis:
            nome_norm = norm_text(nome)
            if nome_norm in aba_norm:
                return aba

    raise ValueError(
        f"Não encontrei aba compatível em {excel_path.name}.\n"
        f"Abas disponíveis: {abas}\n"
        f"Nomes esperados: {nomes_possiveis}"
    )


def carregar_populacao_xlsx(path):
    abas = {
        "RJ": achar_aba(path, ["População Interpolada RJ", "Populacao Interpolada RJ", "RJ"]),
        "SP": achar_aba(path, ["População Interpolada SP", "Populacao Interpolada SP", "SP"]),
    }

    partes = []

    for estado, sheet in abas.items():
        df = pd.read_excel(path, sheet_name=sheet)
        df.columns = [norm_text(c).lower() for c in df.columns]

        col_cod = next(
            c for c in df.columns
            if ("cod" in c and "mun" in c) or c in ["cod_mun", "codmun", "codigo"]
        )

        col_mun = next(
            c for c in df.columns
            if "municipio" in c or "munic" in c
        )

        col_ano = next(
            c for c in df.columns
            if c == "ano"
        )

        col_pop = next(
            c for c in df.columns
            if "pop" in c
        )

        tmp = df[[col_cod, col_mun, col_ano, col_pop]].copy()
        tmp.columns = ["cod_mun6", "municipio", "ano", "populacao"]

        tmp["estado"] = estado
        tmp["arquivo"] = Path(path).name
        tmp["tipo_arquivo"] = "populacao_xlsx"
        tmp["cod_mun6"] = tmp["cod_mun6"].apply(cod6_from_any)
        tmp["municipio"] = tmp["municipio"].apply(norm_text)
        tmp["ano"] = pd.to_numeric(tmp["ano"], errors="coerce").astype("Int64")
        tmp["populacao"] = tmp["populacao"].apply(clean_number)

        partes.append(tmp)

    out = pd.concat(partes, ignore_index=True)
    out = out[out["ano"].isin(ANOS_POPULACAO)].copy()

    return out[
        [
            "arquivo",
            "tipo_arquivo",
            "estado",
            "cod_mun6",
            "municipio",
            "ano",
            "populacao",
        ]
    ]


def carregar_popnova(path):
    abas = {
        "RJ": achar_aba(path, ["POP-SIDRA-RJ", "POP SIDRA RJ", "RJ"]),
        "SP": achar_aba(path, ["POP-SIDRA-SP", "POP SIDRA SP", "SP"]),
    }

    partes = []

    for estado, sheet in abas.items():
        df = pd.read_excel(path, sheet_name=sheet)

        colunas_norm = {norm_text(c).lower(): c for c in df.columns}

        col_cod = None
        for c in df.columns:
            cn = norm_text(c).lower()
            if cn in ["cod", "codigo", "cod mun", "cod municipio"] or "cod" in cn:
                col_cod = c
                break

        col_mun = None
        for c in df.columns:
            cn = norm_text(c).lower()
            if "municipio" in cn or "uf e municipio" in cn:
                col_mun = c
                break

        if col_cod is None or col_mun is None:
            raise ValueError(
                f"Não consegui identificar colunas de código/município na aba {sheet} de {path.name}.\n"
                f"Colunas disponíveis: {list(df.columns)}"
            )

        anos_cols = [
            c for c in df.columns
            if str(c).isdigit() and int(c) in ANOS_POPNOVA
        ]

        if len(anos_cols) == 0:
            raise ValueError(
                f"Não encontrei colunas de anos na aba {sheet} de {path.name}.\n"
                f"Colunas disponíveis: {list(df.columns)}"
            )

        tmp = df[[col_cod, col_mun] + anos_cols].copy()

        tmp = tmp.melt(
            id_vars=[col_cod, col_mun],
            value_vars=anos_cols,
            var_name="ano",
            value_name="populacao",
        )

        tmp = tmp.rename(
            columns={
                col_cod: "cod_mun7",
                col_mun: "municipio",
            }
        )

        tmp["estado"] = estado
        tmp["arquivo"] = Path(path).name
        tmp["tipo_arquivo"] = "popnova"
        tmp["cod_mun6"] = tmp["cod_mun7"].apply(cod6_from_any)
        tmp["municipio"] = tmp["municipio"].apply(norm_text)
        tmp["ano"] = pd.to_numeric(tmp["ano"], errors="coerce").astype("Int64")
        tmp["populacao"] = tmp["populacao"].apply(clean_number)

        partes.append(
            tmp[
                [
                    "arquivo",
                    "tipo_arquivo",
                    "estado",
                    "cod_mun6",
                    "municipio",
                    "ano",
                    "populacao",
                ]
            ]
        )

    return pd.concat(partes, ignore_index=True)


def ler_csv_flex(path):
    encodings = ["utf-8-sig", "latin1", "cp1252"]

    ultimo_erro = None

    for enc in encodings:
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=enc)
        except Exception as e:
            ultimo_erro = e

    raise ultimo_erro


def carregar_casos(path, estado):
    df = ler_csv_flex(path)

    col_mun = None
    for c in df.columns:
        cn = norm_text(c).lower()
        if "municipio" in cn and ("notificacao" in cn or "residencia" in cn or "munic" in cn):
            col_mun = c
            break

    if col_mun is None:
        for c in df.columns:
            cn = norm_text(c).lower()
            if "municipio" in cn:
                col_mun = c
                break

    if col_mun is None:
        raise ValueError(
            f"Não consegui identificar a coluna de município em {path.name}.\n"
            f"Colunas disponíveis: {list(df.columns)}"
        )

    anos_cols = [
        c for c in df.columns
        if str(c).isdigit() and int(c) in ANOS_CASOS
    ]

    if len(anos_cols) == 0:
        raise ValueError(
            f"Não encontrei colunas de anos no arquivo {path.name}.\n"
            f"Colunas disponíveis: {list(df.columns)}"
        )

    tmp = df[[col_mun] + anos_cols].copy()

    tmp["cod_mun6"] = tmp[col_mun].astype(str).str.extract(r"^(\d{6})")

    tmp["municipio"] = (
        tmp[col_mun]
        .astype(str)
        .str.replace(r"^\d{6}\s*", "", regex=True)
        .apply(norm_text)
    )

    long = tmp.melt(
        id_vars=["cod_mun6", "municipio"],
        value_vars=anos_cols,
        var_name="ano",
        value_name="casos",
    )

    long["estado"] = estado
    long["ano"] = pd.to_numeric(long["ano"], errors="coerce").astype("Int64")
    long["casos"] = long["casos"].apply(clean_number).fillna(0).astype(int)

    return long[["estado", "cod_mun6", "municipio", "ano", "casos"]]


def montar_casos():
    sp = carregar_casos(ARQ_CASOS_SP, "SP")
    rj = carregar_casos(ARQ_CASOS_RJ, "RJ")

    return pd.concat([sp, rj], ignore_index=True)


def auditar_pop(df_pop, df_casos=None):
    df = df_pop.copy()

    df = df.sort_values(
        ["arquivo", "estado", "cod_mun6", "ano"]
    ).reset_index(drop=True)

    grupo = ["arquivo", "estado", "cod_mun6"]

    df["pop_anterior"] = df.groupby(grupo)["populacao"].shift(1)
    df["ano_anterior"] = df.groupby(grupo)["ano"].shift(1)

    df["var_pop_abs"] = df["populacao"] - df["pop_anterior"]
    df["var_pop_pct"] = (df["populacao"] / df["pop_anterior"] - 1) * 100

    delta_ano = df["ano"] - df["ano_anterior"]

    df["cagr_pct"] = np.where(
        (df["pop_anterior"] > 0)
        & (df["populacao"] > 0)
        & (delta_ano > 0),
        ((df["populacao"] / df["pop_anterior"]) ** (1 / delta_ano) - 1) * 100,
        np.nan,
    )

    df["flag_pop_invalida"] = df["populacao"].isna() | (df["populacao"] <= 0)

    df["flag_variacao_anual"] = (
        df["var_pop_pct"].abs() > LIMIAR_VARIACAO_ANUAL_PCT
    ).fillna(False)

    df["flag_cagr_base"] = (
        df["cagr_pct"].abs() > LIMIAR_CAGR_PCT
    ).fillna(False)

    usar_cagr = np.where(
        df["tipo_arquivo"].eq("populacao_xlsx"),
        USAR_CAGR_POPULACAO_XLSX,
        USAR_CAGR_POPNOVA,
    )

    df["flag_cagr"] = usar_cagr & df["flag_cagr_base"]

    df["flag_pequenos_numeros_ano"] = (
        df["populacao"] < LIMIAR_POP_PEQUENA
    ).fillna(False)

    if df_casos is not None and USAR_ALERTA_INCIDENCIA:
        df = df.merge(
            df_casos[["estado", "cod_mun6", "ano", "casos"]],
            on=["estado", "cod_mun6", "ano"],
            how="left",
        )

        df["casos"] = df["casos"].fillna(0).astype(int)
        df["incidencia_100mil"] = df["casos"] / df["populacao"] * 100_000

        df["flag_incidencia_extrema"] = (
            (df["incidencia_100mil"] > LIMIAR_INCIDENCIA_EXTREMA)
            & (df["casos"] >= MIN_CASOS_INCIDENCIA_EXTREMA)
            & (df["populacao"] >= MIN_POP_INCIDENCIA_EXTREMA)
        ).fillna(False)

    else:
        df["casos"] = np.nan
        df["incidencia_100mil"] = np.nan
        df["flag_incidencia_extrema"] = False

    df["flag_problema_pop"] = (
        df["flag_pop_invalida"]
        | df["flag_variacao_anual"]
        | df["flag_cagr"]
        | df["flag_incidencia_extrema"]
    )

    return df


def motivos_do_grupo(g):
    motivos = []

    if g["flag_pop_invalida"].any():
        motivos.append("população ausente, zero ou negativa")

    if g["flag_variacao_anual"].any():
        motivos.append(
            f"variação anual da população > {LIMIAR_VARIACAO_ANUAL_PCT}%"
        )

    if g["flag_cagr"].any():
        motivos.append(
            f"CAGR absoluto > {LIMIAR_CAGR_PCT}% ao ano"
        )

    if g["flag_incidencia_extrema"].any():
        motivos.append(
            f"incidência > {LIMIAR_INCIDENCIA_EXTREMA}/100 mil "
            f"com casos ≥ {MIN_CASOS_INCIDENCIA_EXTREMA}"
        )

    return "; ".join(motivos)


def compactar_alertas_problema(df):
    base = df[df["flag_problema_pop"]].copy()

    linhas = []

    for keys, g in base.groupby(
        ["arquivo", "estado", "cod_mun6", "municipio"],
        dropna=False,
    ):
        linhas.append(
            {
                "arquivo": keys[0],
                "estado": keys[1],
                "cod_mun6": keys[2],
                "municipio": keys[3],
                "n_anos_alerta": g["ano"].nunique(),
                "anos_alerta": ", ".join(
                    map(str, sorted(g["ano"].dropna().astype(int).unique()))
                ),
                "motivos": motivos_do_grupo(g),
                "menor_populacao": g["populacao"].min(),
                "maior_populacao": g["populacao"].max(),
                "maior_var_pop_pct_abs": g["var_pop_pct"].abs().max(),
                "maior_cagr_pct_abs": g["cagr_pct"].abs().max(),
                "maior_incidencia_100mil": g["incidencia_100mil"].max(),
                "maior_numero_casos": g["casos"].max(),
            }
        )

    if not linhas:
        return pd.DataFrame(
            columns=[
                "arquivo",
                "estado",
                "cod_mun6",
                "municipio",
                "n_anos_alerta",
                "anos_alerta",
                "motivos",
                "menor_populacao",
                "maior_populacao",
                "maior_var_pop_pct_abs",
                "maior_cagr_pct_abs",
                "maior_incidencia_100mil",
                "maior_numero_casos",
            ]
        )

    return (
        pd.DataFrame(linhas)
        .sort_values(["arquivo", "estado", "municipio"])
        .reset_index(drop=True)
    )


def compactar_pequenos_numeros(df):
    base = df[df["flag_pequenos_numeros_ano"]].copy()

    linhas = []

    for keys, g in base.groupby(
        ["arquivo", "estado", "cod_mun6", "municipio"],
        dropna=False,
    ):
        linhas.append(
            {
                "arquivo": keys[0],
                "estado": keys[1],
                "cod_mun6": keys[2],
                "municipio": keys[3],
                "n_anos_pop_menor_10000": g["ano"].nunique(),
                "anos_pop_menor_10000": ", ".join(
                    map(str, sorted(g["ano"].dropna().astype(int).unique()))
                ),
                "menor_populacao": g["populacao"].min(),
                "maior_populacao": g["populacao"].max(),
            }
        )

    if not linhas:
        return pd.DataFrame(
            columns=[
                "arquivo",
                "estado",
                "cod_mun6",
                "municipio",
                "n_anos_pop_menor_10000",
                "anos_pop_menor_10000",
                "menor_populacao",
                "maior_populacao",
            ]
        )

    return (
        pd.DataFrame(linhas)
        .sort_values(["arquivo", "estado", "municipio"])
        .reset_index(drop=True)
    )


def resumo_por_arquivo(auditoria, problemas, pequenos):
    linhas = []

    for arq in sorted(auditoria["arquivo"].dropna().unique()):
        for estado in sorted(
            auditoria.loc[auditoria["arquivo"].eq(arq), "estado"]
            .dropna()
            .unique()
        ):
            linhas.append(
                {
                    "arquivo": arq,
                    "estado": estado,
                    "municipios_auditados": auditoria[
                        (auditoria["arquivo"].eq(arq))
                        & (auditoria["estado"].eq(estado))
                    ]["cod_mun6"].nunique(),
                    "municipios_com_possivel_problema_pop": problemas[
                        (problemas["arquivo"].eq(arq))
                        & (problemas["estado"].eq(estado))
                    ]["cod_mun6"].nunique(),
                    "municipios_com_pequenos_numeros": pequenos[
                        (pequenos["arquivo"].eq(arq))
                        & (pequenos["estado"].eq(estado))
                    ]["cod_mun6"].nunique(),
                }
            )

    return pd.DataFrame(linhas)


populacao = carregar_populacao_xlsx(ARQ_POPULACAO)
popnova = carregar_popnova(ARQ_POPNOVA)
casos = montar_casos()

aud_populacao = auditar_pop(populacao, casos)
aud_popnova = auditar_pop(popnova, casos)

auditoria_completa = pd.concat(
    [aud_populacao, aud_popnova],
    ignore_index=True,
)

problemas_pop = compactar_alertas_problema(auditoria_completa)
pequenos_numeros = compactar_pequenos_numeros(auditoria_completa)
resumo = resumo_por_arquivo(
    auditoria_completa,
    problemas_pop,
    pequenos_numeros,
)

with pd.ExcelWriter(OUTFILE, engine="openpyxl") as writer:
    resumo.to_excel(writer, sheet_name="Resumo", index=False)
    problemas_pop.to_excel(writer, sheet_name="Problemas_populacao", index=False)
    pequenos_numeros.to_excel(writer, sheet_name="Pequenos_numeros", index=False)
    auditoria_completa.to_excel(writer, sheet_name="Auditoria_ano_a_ano", index=False)

print("\nArquivo salvo em:")
print(OUTFILE)

print("\nResumo:")
display(resumo)

print("\nPossíveis problemas populacionais:")
display(problemas_pop)

print("\nMunicípios sujeitos à lei dos pequenos números:")
display(pequenos_numeros)

Arquivos encontrados:
População: C:\Users\Vitor\MESTRADO-PROJETO\População\População.xlsx
POPNOVA: C:\Users\Vitor\MESTRADO-PROJETO\População\POPNOVA_fim.xlsx
Casos SP: C:\Users\Vitor\MESTRADO-PROJETO\Casos-TB-2001-2024\Casos-TB-SP-NOVO.csv
Casos RJ: C:\Users\Vitor\MESTRADO-PROJETO\Casos-TB-2001-2024\Casos-TB-RJ-NOVO.csv
Saída: C:\Users\Vitor\MESTRADO-PROJETO\População\auditoria_populacao_popnova.xlsx

Arquivo salvo em:
C:\Users\Vitor\MESTRADO-PROJETO\População\auditoria_populacao_popnova.xlsx

Resumo:


,arquivo,estado,municipios_auditados,municipios_com_possivel_problema_pop,municipios_com_pequenos_numeros
0,POPNOVA_fim.xlsx,RJ,92,10,11
1,POPNOVA_fim.xlsx,SP,645,13,300
2,População.xlsx,RJ,92,1,10
3,População.xlsx,SP,645,10,305



Possíveis problemas populacionais:


,arquivo,estado,cod_mun6,municipio,n_anos_alerta,anos_alerta,motivos,menor_populacao,maior_populacao,maior_var_pop_pct_abs,maior_cagr_pct_abs,maior_incidencia_100mil,maior_numero_casos
0,POPNOVA_fim.xlsx,RJ,330010,ANGRA DOS REIS,1,2022,variação anual da população > 20%,167434,167434,20.334394,20.334394,68.683780,115
1,POPNOVA_fim.xlsx,RJ,330100,CAMPOS DOS GOYTACAZES,3,"2022, 2023, 2024",variação anual da população > 20%; incidência ...,13847,519011,512.226475,512.226475,3488.120170,556
2,POPNOVA_fim.xlsx,RJ,330110,CANTAGALO,3,"2022, 2023, 2024",variação anual da população > 20%,8741,19996,56.648316,56.648316,183.045418,16
3,POPNOVA_fim.xlsx,RJ,330093,CARAPEBUS,3,"2022, 2023, 2024",variação anual da população > 20%,14325,483540,2768.141645,2768.141645,20.942408,5
4,POPNOVA_fim.xlsx,RJ,330115,CARDOSO MOREIRA,1,2022,variação anual da população > 20%,19390,19390,51.271649,51.271649,5.157298,1
5,POPNOVA_fim.xlsx,RJ,330120,CARMO,1,2022,variação anual da população > 20%,12958,12958,32.373049,32.373049,84.889643,11
6,POPNOVA_fim.xlsx,RJ,330130,CASIMIRO DE ABREU,3,"2022, 2023, 2024",variação anual da população > 20%,17198,48563,68.042796,68.042796,98.848703,21
7,POPNOVA_fim.xlsx,RJ,330095,COMENDADOR LEVY GASPARIAN,3,"2022, 2023, 2024",variação anual da população > 20%,9044,46110,436.786962,436.786962,44.228218,4
8,POPNOVA_fim.xlsx,RJ,330360,PARACAMBI,1,2022,variação anual da população > 20%,41375,41375,22.070706,22.070706,50.755287,21
9,POPNOVA_fim.xlsx,RJ,330452,RIO DAS OSTRAS,2,"2007, 2008",variação anual da população > 20%,67396,91085,35.148970,35.148970,80.123449,54



Municípios sujeitos à lei dos pequenos números:


,arquivo,estado,cod_mun6,municipio,n_anos_pop_menor_10000,anos_pop_menor_10000,menor_populacao,maior_populacao
0,POPNOVA_fim.xlsx,RJ,330015,APERIBE,9,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",8205,9556
1,POPNOVA_fim.xlsx,RJ,330110,CANTAGALO,1,2022,8741,8741
2,POPNOVA_fim.xlsx,RJ,330093,CARAPEBUS,4,"2001, 2002, 2003, 2004",8882,9951
3,POPNOVA_fim.xlsx,RJ,330095,COMENDADOR LEVY GASPARIAN,22,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",8026,9044
4,POPNOVA_fim.xlsx,RJ,330230,LAJE DO MURIAE,24,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",7217,8238
...,...,...,...,...,...,...,...,...
621,População.xlsx,SP,355635,VARGEM,12,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007...",6975,9987
622,População.xlsx,SP,355645,VARGEM GRANDE PAULISTA,2,"2023, 2024",8312,9348
623,População.xlsx,SP,355690,VISTA ALEGRE DO ALTO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007...",4754,8333
624,População.xlsx,SP,355695,VITORIA BRASIL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007...",1675,1804


In [3]:
# ============================================================
# Separar tabelas por arquivo
# ============================================================

nome_populacao = ARQ_POPULACAO.name
nome_popnova = ARQ_POPNOVA.name

resumo_populacao = resumo[resumo["arquivo"].eq(nome_populacao)].copy()
resumo_popnova = resumo[resumo["arquivo"].eq(nome_popnova)].copy()

problemas_populacao = problemas_pop[problemas_pop["arquivo"].eq(nome_populacao)].copy()
problemas_popnova = problemas_pop[problemas_pop["arquivo"].eq(nome_popnova)].copy()

pequenos_populacao = pequenos_numeros[pequenos_numeros["arquivo"].eq(nome_populacao)].copy()
pequenos_popnova = pequenos_numeros[pequenos_numeros["arquivo"].eq(nome_popnova)].copy()

auditoria_populacao = auditoria_completa[
    auditoria_completa["arquivo"].eq(nome_populacao)
].copy()

auditoria_popnova = auditoria_completa[
    auditoria_completa["arquivo"].eq(nome_popnova)
].copy()


# ============================================================
# Salvar Excel com abas separadas por arquivo
# ============================================================

with pd.ExcelWriter(OUTFILE, engine="openpyxl") as writer:
    
    # Resumos
    resumo.to_excel(writer, sheet_name="Resumo_geral", index=False)
    resumo_populacao.to_excel(writer, sheet_name="Resumo_Populacao", index=False)
    resumo_popnova.to_excel(writer, sheet_name="Resumo_POPNOVA", index=False)

    # Problemas populacionais
    problemas_populacao.to_excel(writer, sheet_name="Problemas_Populacao", index=False)
    problemas_popnova.to_excel(writer, sheet_name="Problemas_POPNOVA", index=False)

    # Lei dos pequenos números
    pequenos_populacao.to_excel(writer, sheet_name="Pequenos_Populacao", index=False)
    pequenos_popnova.to_excel(writer, sheet_name="Pequenos_POPNOVA", index=False)

    # Auditoria ano a ano
    auditoria_populacao.to_excel(writer, sheet_name="Auditoria_Populacao", index=False)
    auditoria_popnova.to_excel(writer, sheet_name="Auditoria_POPNOVA", index=False)


print("\nArquivo salvo em:")
print(OUTFILE)

print("\nResumo - População.xlsx:")
display(resumo_populacao)

print("\nPossíveis problemas populacionais - População.xlsx:")
display(problemas_populacao)

print("\nPequenos números - População.xlsx:")
display(pequenos_populacao)

print("\nResumo - POPNOVA_fim.xlsx:")
display(resumo_popnova)

print("\nPossíveis problemas populacionais - POPNOVA_fim.xlsx:")
display(problemas_popnova)

print("\nPequenos números - POPNOVA_fim.xlsx:")
display(pequenos_popnova)


Arquivo salvo em:
C:\Users\Vitor\MESTRADO-PROJETO\População\auditoria_populacao_popnova.xlsx

Resumo - População.xlsx:


,arquivo,estado,municipios_auditados,municipios_com_possivel_problema_pop,municipios_com_pequenos_numeros
2,População.xlsx,RJ,92,1,10
3,População.xlsx,SP,645,10,305



Possíveis problemas populacionais - População.xlsx:


,arquivo,estado,cod_mun6,municipio,n_anos_alerta,anos_alerta,motivos,menor_populacao,maior_populacao,maior_var_pop_pct_abs,maior_cagr_pct_abs,maior_incidencia_100mil,maior_numero_casos
23,População.xlsx,RJ,330100,CAMPOS DOS GOYTACAZES,21,"2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011...",incidência > 1000/100 mil com casos ≥ 20,10304,13930,4.428904,4.428904,4003.456221,556
24,População.xlsx,SP,350250,APARECIDA,14,"2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018...",CAGR absoluto > 12% ao ano,2856,29270,16.397455,16.397455,735.294118,24
25,População.xlsx,SP,350260,APARECIDA D OESTE,14,"2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018...",CAGR absoluto > 12% ao ano,5253,45381,18.048914,18.048914,23.148148,2
26,População.xlsx,SP,350970,CAMPOS DO JORDAO,3,"2001, 2002, 2003",incidência > 1000/100 mil com casos ≥ 20,44594,45285,0.772846,0.772846,1722.204781,768
27,População.xlsx,SP,351530,ESTRELA DO NORTE,10,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",CAGR absoluto > 12% ao ano,2942,8208,12.097429,12.097429,13.653741,1
28,População.xlsx,SP,354540,SALTO GRANDE,14,"2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018...",variação anual da população > 20%; CAGR absolu...,11029,211602,25.516057,25.516057,72.536041,9
29,População.xlsx,SP,354740,SANTA RITA D OESTE,24,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",variação anual da população > 20%; CAGR absolu...,1872,26478,25.681391,25.681391,88.417330,2
30,População.xlsx,SP,354750,SANTA RITA DO PASSA QUATRO,24,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",variação anual da população > 20%; CAGR absolu...,2543,36306,20.934372,20.934372,188.273265,9
31,População.xlsx,SP,355070,SAO SEBASTIAO,14,"2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018...",CAGR absoluto > 12% ao ano,7534,62812,15.056029,15.056029,1327.316167,100
32,População.xlsx,SP,355080,SAO SEBASTIAO DA GRAMA,14,"2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018...",CAGR absoluto > 12% ao ano,14185,112154,17.243679,17.243679,15.913431,5



Pequenos números - População.xlsx:


...

,arquivo,estado,municipios_auditados,municipios_com_possivel_problema_pop,municipios_com_pequenos_numeros
0,POPNOVA_fim.xlsx,RJ,92,10,11
1,POPNOVA_fim.xlsx,SP,645,13,300



Possíveis problemas populacionais - POPNOVA_fim.xlsx:


,arquivo,estado,cod_mun6,municipio,n_anos_alerta,anos_alerta,motivos,menor_populacao,maior_populacao,maior_var_pop_pct_abs,maior_cagr_pct_abs,maior_incidencia_100mil,maior_numero_casos
0,POPNOVA_fim.xlsx,RJ,330010,ANGRA DOS REIS,1,2022,variação anual da população > 20%,167434,167434,20.334394,20.334394,68.683780,115
1,POPNOVA_fim.xlsx,RJ,330100,CAMPOS DOS GOYTACAZES,3,"2022, 2023, 2024",variação anual da população > 20%; incidência ...,13847,519011,512.226475,512.226475,3488.120170,556
2,POPNOVA_fim.xlsx,RJ,330110,CANTAGALO,3,"2022, 2023, 2024",variação anual da população > 20%,8741,19996,56.648316,56.648316,183.045418,16
3,POPNOVA_fim.xlsx,RJ,330093,CARAPEBUS,3,"2022, 2023, 2024",variação anual da população > 20%,14325,483540,2768.141645,2768.141645,20.942408,5
4,POPNOVA_fim.xlsx,RJ,330115,CARDOSO MOREIRA,1,2022,variação anual da população > 20%,19390,19390,51.271649,51.271649,5.157298,1
5,POPNOVA_fim.xlsx,RJ,330120,CARMO,1,2022,variação anual da população > 20%,12958,12958,32.373049,32.373049,84.889643,11
6,POPNOVA_fim.xlsx,RJ,330130,CASIMIRO DE ABREU,3,"2022, 2023, 2024",variação anual da população > 20%,17198,48563,68.042796,68.042796,98.848703,21
7,POPNOVA_fim.xlsx,RJ,330095,COMENDADOR LEVY GASPARIAN,3,"2022, 2023, 2024",variação anual da população > 20%,9044,46110,436.786962,436.786962,44.228218,4
8,POPNOVA_fim.xlsx,RJ,330360,PARACAMBI,1,2022,variação anual da população > 20%,41375,41375,22.070706,22.070706,50.755287,21
9,POPNOVA_fim.xlsx,RJ,330452,RIO DAS OSTRAS,2,"2007, 2008",variação anual da população > 20%,67396,91085,35.148970,35.148970,80.123449,54



Pequenos números - POPNOVA_fim.xlsx:


,arquivo,estado,cod_mun6,municipio,n_anos_pop_menor_10000,anos_pop_menor_10000,menor_populacao,maior_populacao
0,POPNOVA_fim.xlsx,RJ,330015,APERIBE,9,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",8205,9556
1,POPNOVA_fim.xlsx,RJ,330110,CANTAGALO,1,2022,8741,8741
2,POPNOVA_fim.xlsx,RJ,330093,CARAPEBUS,4,"2001, 2002, 2003, 2004",8882,9951
3,POPNOVA_fim.xlsx,RJ,330095,COMENDADOR LEVY GASPARIAN,22,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",8026,9044
4,POPNOVA_fim.xlsx,RJ,330230,LAJE DO MURIAE,24,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",7217,8238
...,...,...,...,...,...,...,...,...
306,POPNOVA_fim.xlsx,SP,355635,VARGEM,15,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",7092,9854
307,POPNOVA_fim.xlsx,SP,355660,VERA CRUZ,1,2009,9952,9952
308,POPNOVA_fim.xlsx,SP,355690,VISTA ALEGRE DO ALTO,24,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",4894,9163
309,POPNOVA_fim.xlsx,SP,355695,VITORIA BRASIL,24,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008...",1662,1852


# Considerando esses resultados, utilizaremos o arquivo "População.xlsx" para realizar a análise exploratória (Depois de corrigir os problemas de população!).